# Gradient Descent

**Topic:** Optimization in Machine Learning

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from ipywidgets import IntSlider, FloatSlider, Dropdown, ToggleButtons, Output, HBox, VBox, Label
from IPython.display import display, clear_output
from scipy import stats
from sklearn.datasets import make_classification, make_regression, make_circles, make_moons
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** how gradient descent finds the minimum of a loss function step by step
- **Explain** the effect of the learning rate on convergence speed and stability
- **Interpret** the difference between batch, stochastic, and mini-batch gradient descent

> **Tip:** Set the learning rate very high (above 0.85) and watch the ball overshoot. Then set it very low (below 0.05) and watch it inch forward. The sweet spot in the middle is what makes modern deep learning work.

---
## How we got here

In **07_loss_functions.ipynb** we established what the model is trying to minimize. Now we ask: *how* does it actually minimize it?

This connects directly to:
- **math/calculus/06_gradient_and_gradient_descent.ipynb** — you built the mathematical intuition for gradients and took your first symbolic descent steps
- **math/calculus/09_optimization.ipynb** — you explored the landscape of optimization problems

Here you see those mathematical tools driving actual model training.

---
## Why this matters for data science

Every major machine learning algorithm — linear regression, logistic regression, neural networks, gradient boosting — uses some form of gradient descent under the hood. Understanding it means you can diagnose training failures, choose the right learning rate, and interpret loss curves during training.

When you see "loss not decreasing" in a training log, you will know exactly what to look for.

---
## Try it yourself

In [ ]:
np.random.seed(42)

# Elongated bowl: steep along w0, gentle along w1 — the asymmetry bends the path.
A, B = 1.0, 0.25
W0_STAR, W1_STAR = 3.0, -2.0
def loss_fn(w0, w1): return A*(w0 - W0_STAR)**2 + B*(w1 - W1_STAR)**2 + 1.0
def grad_fn(w0, w1): return np.array([2*A*(w0 - W0_STAR), 2*B*(w1 - W1_STAR)])

w0_grid = np.linspace(-2, 8, 60)
w1_grid = np.linspace(-7, 3, 60)
W0, W1 = np.meshgrid(w0_grid, w1_grid)
Z = loss_fn(W0, W1)

out = Output()

lr_slider = FloatSlider(value=0.1, min=0.01, max=1.10, step=0.05,
    description="Learning rate:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="420px"))

variant_dd = Dropdown(
    options=["Batch gradient descent", "Stochastic gradient descent", "Mini-batch (size 8)"],
    value="Batch gradient descent",
    description="Variant:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="420px"))

caption = widgets.HTML(layout=widgets.Layout(width="660px"))

def simulate_gd(lr, variant, n_steps=40):
    w = np.array([-1.0, 2.0])        # start far from the minimum
    path = [w.copy()]
    for _ in range(n_steps):
        noise = np.zeros(2)
        if variant.startswith("Stochastic"):
            noise = np.random.normal(0, 0.8, size=2)
        elif variant.startswith("Mini"):
            noise = np.random.normal(0, 0.3, size=2)
        g = grad_fn(w[0], w[1]) + noise
        w = w - lr * g
        path.append(w.copy())
        if variant.startswith("Batch") and np.linalg.norm(g) < 1e-4:
            break
    return np.array(path)

def render(lr, variant):
    path = simulate_gd(lr, variant)
    z_path = loss_fn(path[:, 0], path[:, 1])
    final = path[-1]
    dist = float(np.hypot(final[0] - W0_STAR, final[1] - W1_STAR))
    LIFT = 0.4

    # ---------- 3D SURFACE ----------
    surface = go.Surface(
        x=w0_grid, y=w1_grid, z=Z,
        colorscale=[[0, PALETTE["primary"]], [1, PALETTE["muted"]]],
        opacity=0.72, showscale=False,
        contours={"z": {"show": True, "usecolormap": True, "project_z": True}})
    fig3d = go.Figure(data=[
        surface,
        go.Scatter3d(x=path[:, 0], y=path[:, 1], z=z_path + LIFT,
            mode="lines+markers", line=dict(color=PALETTE["secondary"], width=5),
            marker=dict(size=4, color=PALETTE["secondary"]), name="Descent path"),
        go.Scatter3d(x=[path[0, 0]], y=[path[0, 1]], z=[z_path[0] + LIFT],
            mode="markers", marker=dict(size=7, color=PALETTE["accent"]), name="Start"),
        go.Scatter3d(x=[W0_STAR], y=[W1_STAR], z=[loss_fn(W0_STAR, W1_STAR)],
            mode="markers", marker=dict(size=7, color=PALETTE["muted"], symbol="diamond"),
            name="True minimum"),
        go.Scatter3d(x=[final[0]], y=[final[1]], z=[z_path[-1] + LIFT],
            mode="markers", marker=dict(size=9, color=PALETTE["secondary"], symbol="diamond"),
            name=f"Stopped ({dist:.2f} from min)"),
    ])
    lay3d = base_layout(
        title=f"{variant} | lr={lr:.2f} | steps={len(path)-1} | final loss={z_path[-1]:.3f}",
        xaxis_title="w0", yaxis_title="w1")
    lay3d.update(height=520, scene=dict(
        xaxis=dict(title="w₀", range=[-2, 8]),
        yaxis=dict(title="w₁", range=[-7, 3]),
        zaxis=dict(title="Loss", range=[0, 35]),
        camera=dict(eye=dict(x=1.7, y=1.7, z=1.1))))
    fig3d.update_layout(lay3d)

    # ---------- 2D CONTOUR (top-down) ----------
    fig_contour = go.Figure(data=[
        go.Contour(x=w0_grid, y=w1_grid, z=Z, showscale=False,
            colorscale=[[0, PALETTE["primary"]], [1, PALETTE["muted"]]],
            contours=dict(showlines=True), opacity=0.9),
        go.Scatter(x=path[:, 0], y=path[:, 1], mode="lines+markers",
            line=dict(color=PALETTE["secondary"], width=2),
            marker=dict(size=6, color=PALETTE["secondary"]), name="Path"),
        go.Scatter(x=[path[0, 0]], y=[path[0, 1]], mode="markers",
            marker=dict(size=11, color=PALETTE["accent"]), name="Start"),
        go.Scatter(x=[W0_STAR], y=[W1_STAR], mode="markers",
            marker=dict(size=13, color=PALETTE["muted"], symbol="x",
                        line=dict(width=2)), name="Minimum"),
    ])
    lay_c = base_layout(title="Top-down view (contours)",
        xaxis_title="w₀", yaxis_title="w₁")
    lay_c.update(width=420, height=380,
        xaxis=dict(range=[-2, 8]), yaxis=dict(range=[-7, 3]))
    fig_contour.update_layout(lay_c)

    # ---------- LOSS vs ITERATION (the training-log curve) ----------
    fig_loss = go.Figure(data=[
        go.Scatter(x=list(range(len(z_path))), y=z_path, mode="lines+markers",
            line=dict(color=PALETTE["primary"], width=2.5),
            marker=dict(size=5, color=PALETTE["primary"]), name="Loss"),
        go.Scatter(x=[0, len(z_path)-1], y=[loss_fn(W0_STAR, W1_STAR)]*2,
            mode="lines", line=dict(color=PALETTE["muted"], width=1.5, dash="dash"),
            name="Best possible loss"),
    ])
    lay_l = base_layout(title="Loss vs. iteration (training log)",
        xaxis_title="Step", yaxis_title="Loss")
    lay_l.update(width=420, height=380, yaxis=dict(range=[0, max(35, z_path.max()*1.1)]))
    fig_loss.update_layout(lay_l)

    with out:
        clear_output(wait=True)
        display(VBox([
            go.FigureWidget(fig3d),
            HBox([go.FigureWidget(fig_contour), go.FigureWidget(fig_loss)]),
        ]))

    # ---- Dynamic caption ----
    if lr > 1.0:
        regime = ("<b>Diverging.</b> Along the steep w₀ direction each step overshoots farther "
                  "than it started — on the loss curve you'll see it climb instead of fall.")
    elif lr >= 0.5:
        regime = ("<b>Oscillating but converging.</b> The steep w₀ direction overshoots and lands "
                  "on the far side each step — the contour path zig-zags and the loss curve wobbles down.")
    else:
        regime = ("<b>Smooth descent.</b> The path drops fast along steep w₀, then crawls along gentle "
                  "w₁ — that bend in the contour view is the bowl being steeper in one direction.")

    if variant.startswith("Batch"):
        var_note = ("<b>Batch</b> uses the exact gradient, so all three views agree: a clean curve "
                    "into the minimum and a smooth, monotonic loss curve.")
    else:
        var_note = (f"<b>{variant.split(' (')[0]}</b> uses a noisy gradient — the contour path wanders "
                    f"and the loss curve is jagged, ending <i>near</i> the minimum ({dist:.2f} away), not on "
                    "it. Stylized noise model: real SGD noise comes from each step seeing a different "
                    "mini-batch and shrinks near a broad minimum rather than staying constant.")

    caption.value = (f"<div style='font-size:13px; line-height:1.5; padding:6px 2px'>"
                     f"{regime}<br><br>{var_note}</div>")

def on_change(change): render(lr_slider.value, variant_dd.value)
lr_slider.observe(on_change, names="value")
variant_dd.observe(on_change, names="value")
display(VBox([lr_slider, variant_dd, out, caption]))
render(0.1, "Batch gradient descent")

---
## What's happening?

Imagine hiking down a foggy mountain. You cannot see the bottom, but you can always feel which way is downhill and take a step in that direction. After enough steps you reach the valley.

Gradient descent is that process applied to a loss function:
1. Start at a random parameter value
2. Compute the gradient (slope of the loss at that point)
3. Take a step in the direction that reduces the loss
4. Repeat until the gradient is near zero

The **learning rate** controls step size. Too large: you overshoot the minimum and bounce around. Too small: you move so slowly that training takes too long to converge.

| Variant | Samples per step | Speed | Stability | When to use |
|---|---|---|---|---|
| Batch | All training data | Slow | Very stable | Small datasets |
| Stochastic (SGD) | One sample | Fast | Noisy | Online learning, very large data |
| Mini-batch | Batch of 32-256 | Fast | Stable enough | Deep learning (default) |

---
## Real-world example: Training a neural network

A neural network for image classification might have 25 million parameters. Computing the gradient over all 1 million training images at once (batch GD) would take hours per update. Instead, mini-batch gradient descent picks 128 random images, computes the gradient for those, and updates immediately.

The noisy but fast updates average out over thousands of steps. The model converges in hours rather than days — and the noise actually helps it escape shallow local minima.

---
## Key takeaway

> **Gradient descent minimizes the loss by repeatedly moving parameters in the direction that reduces error — and the learning rate controls how far each step moves, with too large causing oscillation and too small causing slow convergence.**

---
*Next up: The No Free Lunch Theorem — why no single algorithm wins on every problem*